In [1]:
# STAGE 1: Load the dataset
# We're using a real retail dataset from UCI - this matters for credibility

import pandas as pd

# Load directly from URL - no manual downloading needed
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx"

# This will take 30-60 seconds - it's a large file
print("Loading dataset... this may take a minute")
df = pd.read_excel(url, sheet_name='Year 2009-2010')

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns")
print(df.head())

Loading dataset... this may take a minute
Loaded 525,461 rows and 8 columns
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


In [2]:
# STAGE 2: Understand what we're actually working with
# A good analyst always looks before they touch

print("=== SHAPE ===")
print(f"{df.shape[0]:,} rows, {df.shape[1]} columns")

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== BASIC STATS ===")
print(df[['Quantity', 'Price']].describe())

print("\n=== SAMPLE INVOICE DATES ===")
print(f"Earliest: {df['InvoiceDate'].min()}")
print(f"Latest: {df['InvoiceDate'].max()}")

print("\n=== TOP 5 COUNTRIES ===")
print(df['Country'].value_counts().head())

=== SHAPE ===
525,461 rows, 8 columns

=== MISSING VALUES ===
Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

=== DATA TYPES ===
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

=== BASIC STATS ===
            Quantity          Price
count  525461.000000  525461.000000
mean       10.337667       4.688834
std       107.424110     146.126914
min     -9600.000000  -53594.360000
25%         1.000000       1.250000
50%         3.000000       2.100000
75%        10.000000       4.210000
max     19152.000000   25111.090000

=== SAMPLE INVOICE DATES ===
Earliest: 2009-12-01 07:45:00
Latest: 2010-12-09 20:01:00

=== TOP 5 COUNTRIES ===
Country
United Ki

In [3]:
# STAGE 2: Data Cleaning
# Every decision here should be defensible in an interview

print("Rows before cleaning:", len(df))

# Step 1: Remove cancellations
# Invoices starting with 'C' are cancellations in this dataset
# Negative quantities are also returns
df_clean = df[~df['Invoice'].astype(str).str.startswith('C')]
df_clean = df_clean[df_clean['Quantity'] > 0]
print(f"After removing cancellations/returns: {len(df_clean):,} rows")

# Step 2: Remove bad prices
# Zero or negative price means bad data or internal transfers
df_clean = df_clean[df_clean['Price'] > 0]
print(f"After removing bad prices: {len(df_clean):,} rows")

# Step 3: UK only - cleaner story, more data
df_clean = df_clean[df_clean['Country'] == 'United Kingdom']
print(f"After filtering to UK only: {len(df_clean):,} rows")

# Step 4: Drop missing descriptions (tiny, safe to drop)
df_clean = df_clean.dropna(subset=['Description'])
print(f"After dropping missing descriptions: {len(df_clean):,} rows")

# Step 5: Create revenue column - this is our target variable
# Revenue = how much money each line item made
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']

# Step 6: Extract date only from timestamp (we'll forecast at daily level)
df_clean['Date'] = df_clean['InvoiceDate'].dt.date

print("\n=== CLEANED DATA SAMPLE ===")
print(df_clean[['Invoice', 'Description', 'Quantity', 'Price', 'Revenue', 'Date']].head())

print("\n=== REVENUE STATS AFTER CLEANING ===")
print(df_clean['Revenue'].describe())

Rows before cleaning: 525461
After removing cancellations/returns: 513,134 rows
After removing bad prices: 511,565 rows
After filtering to UK only: 473,378 rows
After dropping missing descriptions: 473,378 rows

=== CLEANED DATA SAMPLE ===
  Invoice                          Description  Quantity  Price  Revenue  \
0  489434  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   6.95     83.4   
1  489434                   PINK CHERRY LIGHTS        12   6.75     81.0   
2  489434                  WHITE CHERRY LIGHTS        12   6.75     81.0   
3  489434         RECORD FRAME 7" SINGLE SIZE         48   2.10    100.8   
4  489434       STRAWBERRY CERAMIC TRINKET BOX        24   1.25     30.0   

         Date  
0  2009-12-01  
1  2009-12-01  
2  2009-12-01  
3  2009-12-01  
4  2009-12-01  

=== REVENUE STATS AFTER CLEANING ===
count    473378.000000
mean         18.686453
std          89.771935
min           0.001000
25%           3.750000
50%           8.950000
75%          17.000000
max     